In [ ]:
import heapq
from itertools import combinations
from collections import defaultdict
import math


class Assignment:
    def __init__(self, a_id, in1, in2, out, food):
        self.id = f"A{a_id}"
        self.inputs = [int(in1), int(in2)]
        self.out = int(out)
        self.food = food

def compute_depths(assignments, adj_list):
    memo = {}
    def dfs(node):
        if node in memo:
            return memo[node]
        max_depth = 0
        if adj_list[node]:
            for neighbor in adj_list[node]:
                max_depth = max(max_depth, 1 + dfs(neighbor))
        memo[node] = max_depth
        return max_depth

    for a in assignments:
        dfs(a.id)
    return memo


def get_h_cost(uncompleted, depths, g_size):
    """ Calculates the admissible heuristic h(n). """
    if not uncompleted:
        return 0

    # 1. Capacity bound: Minimum days based on group size
    capacity_bound = math.ceil(len(uncompleted) / g_size)

    # 2. Critical path bound: Longest remaining dependency chain
    critical_path_bound = max(depths[task] for task in uncompleted) + 1

    return max(capacity_bound, critical_path_bound)


def solve_a_star(assignments, food_costs, initial_inputs, g_size):
    out_to_assign = {a.out: a.id for a in assignments}
    assign_map = {a.id: a for a in assignments}

    in_degree = {a.id: 0 for a in assignments}
    adj_list = defaultdict(list)

    for a in assignments:
        for req in a.inputs:
            if req not in initial_inputs:
                if req in out_to_assign:
                    parent = out_to_assign[req]
                    adj_list[parent].append(a.id)
                    in_degree[a.id] += 1

    depths = compute_depths(assignments, adj_list)

    initial_available = frozenset(a.id for a in assignments if in_degree[a.id] == 0)
    all_tasks = frozenset(a.id for a in assignments)

    start_h = get_h_cost(all_tasks, depths, g_size)
    pq = [(start_h, 0, 0, (frozenset(), initial_available, []))]

    visited = set()
    states_explored = 0
    state_counter = 0

    while pq:
        f, g_cost, _, (completed, available, schedule) = heapq.heappop(pq)
        states_explored += 1

        if len(completed) == len(assignments):
            return schedule, states_explored, g_cost

        state_key = (completed, available)
        if state_key in visited:
            continue
        visited.add(state_key)

        available_list = list(available)
        max_pick = min(g_size, len(available_list))

        for k in range(1, max_pick + 1):
            for daily_tasks in combinations(available_list, k):
                daily_tasks_set = frozenset(daily_tasks)

                daily_menu = defaultdict(int)
                day_cost = 0
                for task in daily_tasks:
                    food = assign_map[task].food
                    daily_menu[food] += 1
                    day_cost += food_costs[food]

                new_schedule = schedule + [{
                    "day": g_cost + 1,
                    "tasks": list(daily_tasks),
                    "menu": dict(daily_menu),
                    "cost": day_cost
                }]

                new_completed = completed | daily_tasks_set
                new_available = set(available) - daily_tasks_set

                for task in daily_tasks:
                    for neighbor in adj_list[task]:
                        prereqs_met = True
                        for req in assign_map[neighbor].inputs:
                            if req not in initial_inputs and out_to_assign.get(req) not in new_completed:
                                prereqs_met = False
                                break
                        if prereqs_met:
                            new_available.add(neighbor)

                new_uncompleted = all_tasks - new_completed
                new_h = get_h_cost(new_uncompleted, depths, g_size)
                new_g = g_cost + 1
                new_f = new_g + new_h

                state_counter += 1
                heapq.heappush(pq, (new_f, new_g, state_counter, (new_completed, frozenset(new_available), new_schedule)))

    return None, states_explored, -1



if __name__ == "__main__":
    food_costs = {'TC': 1, 'DF': 1, 'PM': 1, 'GJ': 1}
    g = 2
    initial_inputs = {1, 2, 3, 4, 5, 6}

    raw_assignments = [
        (1, 1, 3, 7, 'TC'), (2, 4, 2, 8, 'TC'), (3, 1, 3, 9, 'TC'),
        (4, 2, 3, 10, 'PM'), (5, 7, 8, 11, 'TC'), (6, 4, 6, 12, 'TC'),
        (7, 6, 9, 13, 'PM'), (8, 10, 5, 14, 'GJ'), (9, 1, 11, 15, 'DF'),
        (10, 3, 12, 16, 'TC'), (11, 15, 16, 17, 'DF')
    ]

    assignments = [Assignment(*data) for data in raw_assignments]
    

    print("\n=== Task 2: A* Search for Optimal Schedule ===")
    optimal_schedule, states_explored, total_days = solve_a_star(assignments, food_costs, initial_inputs, g)

    if optimal_schedule:
        total_a_star_cost = sum(day['cost'] for day in optimal_schedule)

        for day in optimal_schedule:
            print(f"Day-{day['day']}: {', '.join(day['tasks'])}")
            menu_str = ", ".join([f"{v}-{k}" for k, v in day['menu'].items()])
            print(f"Menu: {menu_str} | Cost: {day['cost']}")

        print(f"Total Days: {total_days}")
        print(f"Total Food Cost: {total_a_star_cost}")
        print(f"Total States Explored: {states_explored}")
    else:
        print("No valid schedule found.")


=== Task 2: A* Search for Optimal Schedule ===
Day-1: A3
Menu: 1-TC | Cost: 1
Day-2: A4, A7
Menu: 2-PM | Cost: 2
Day-3: A2, A1
Menu: 2-TC | Cost: 2
Day-4: A6, A5
Menu: 2-TC | Cost: 2
Day-5: A9, A10
Menu: 1-DF, 1-TC | Cost: 2
Day-6: A8, A11
Menu: 1-GJ, 1-DF | Cost: 2
Total Days: 6
Total Food Cost: 11
Total States Explored: 522
